# Προσέξτε το Κενό

---

> Πάνος Λουρίδας, Αναπληρωτής Καθηγητής <br />
> Τμήμα Διοικητικής Επιστήμης και Τεχνολογίας <br />
> Οικονομικό Πανεπιστήμιο Αθηνών <br />
> louridas@aueb.gr


Θα αναπαράξουμε τις αναπαραστάσεις του [Gapminder](http://www.gapminder.org/) για το ΑΕΠ και το προσδόκιμο ζωής, χρησιμοποιώντας την Python. Συγκεκριμένα, θα χρησιμοποιήσουμε το [pandas](http://pandas.pydata.org/) και το [bokeh](http://bokeh.pydata.org/en/latest/). Ο σκοπός είναι να δούμε πώς μπορούμε να κατασκευάσουμε αλληλεπιδραστικές παραστάσεις με ανοιχτά εργαλεία.

Αν σας ενδιαφέρει απλώς το τελικό αποτέλεσμα, πηγαίνετε στο τέλος.

Αλλιώς συνεχίστε παίρνοντας τα πράγματα με τη σειρά τους.

In [1]:
import pandas as pd
import numpy as np
import re

Θα διαβάσουμε τα δεδομένα του κατακεφαλήν ΑΕΠ, σε ισοδύναμη αγοραστική δύναμη, τα οποία τα έχουμε κατεβάσει από το Gapminder και τα έχουμε τοποθετήσει στον κατάλογο (φάκελλο) `gapminder`. Μπορείτε να βρείτε όλα τα δεδομένα του Gapminder στο http://www.gapminder.org/data/. Τα δεδομένα του ΑΕΠ είναι ο δείκτης "Income per person (GDP/capita, PPP$ inflation-adjusted)", πηγή <http://gapm.io/dgdppc>.

In [2]:
gdp_df = pd.read_csv('income_per_person_gdppercapita_ppp_inflation_adjusted.csv')

gdp_df.sample(5, random_state=42)

,country,1800,1801,1802,1803,1804,1805,1806,1807,1808,...,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040
45,Czech Republic,1920,1920,1920,1920,1920,1920,1920,1920,1920,...,42700,43700,44600,45600,46600,47700,48700,49800,50900,52000
137,Portugal,1680,1680,1680,1680,1690,1690,1690,1690,1690,...,35900,36700,37500,38400,39200,40100,41000,41900,42800,43800
76,Indonesia,994,994,995,995,995,995,995,995,996,...,17200,17500,17900,18300,18700,19200,19600,20000,20500,20900
144,Sao Tome and Principe,850,852,853,855,857,858,860,861,863,...,4440,4540,4640,4750,4850,4960,5070,5180,5290,5410
113,Montenegro,1060,1060,1060,1060,1060,1060,1060,1060,1060,...,22800,23300,23800,24400,24900,25500,26000,26600,27200,27800


Παρατηρήστε ότι τα δεδομένα δεν είναι απλώς αριθμοί, καθώς μπορεί να έχουν την κατάληξη `k` (χιλιάδες). Γεκικότερα, τα δεδομένα από το Gapminder μπορεί να έχουν επίσης την κατάληξη `M` (εκατομμύρια) and `B` (δισεκατομμύρια).

Θα γράψουμε μια συνάρτηση για να μετατρέψουμε τα δεδομένα σε απλούς αριθμούς, χωρίς τις καταλήξεις.

In [3]:
suf2num_conv = {
    '' : 1,
    'k': 1e3,
    'm': 1e6,
    'b': 1e9
}

def suf2num(s):
    if s.dtype != object:
        return s
    n_s = s.str.extract(r'([\d\.]+)([kKmMbB]*)', expand=False)
    n_s[0] = n_s[0].astype(float)
    n_s[1] = n_s[1].str.lower()
    n_s['mul'] = n_s[1].map(lambda x : suf2num_conv.get(x, 1))
    return n_s[0] * n_s['mul']

suf2num(gdp_df['1800'])

0       603
1       667
2       715
3      1200
4       618
       ... 
188     682
189     861
190     877
191     663
192     869
Name: 1800, Length: 193, dtype: int64

Θέλουμε να εφαρμόσουμε την μετατροπή σε όλες τις στήλες εκτός από την πρώτη. 

Για να το κάνουμε αυτό, θα εξάγουμε τη στήλη, θα μετατρέψουμε τις υπόλοιπες, και θα βάλουμε πίσω στη θέση της την στήλη που βγάλαμε.

In [4]:
gdp_countries = gdp_df.pop('country')
gdp_df = gdp_df.apply(suf2num, axis=0)
gdp_df['country'] = gdp_countries

Τα δεδομένα περιέχουν και προβολές στο μέλλον, τις οποίες θα τις αφαιρέσουμε (τη στιγμή που γράφονται αυτά οι προβλέψεις αφορού το 2021 και μετά).

In [5]:
gdp_df = gdp_df.drop([str(x) for x in range(2021, 2040+1)], axis=1)
gdp_df

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,country
0,603,603,603,603,603,603,603,603,603,603,...,1840,1810,1780,1750,1740,1800,1870,1950,1970,Afghanistan
1,667,667,667,667,667,668,668,668,668,668,...,10400,10500,10700,11000,11400,11900,12400,13000,13500,Albania
2,715,716,717,718,719,720,721,722,723,724,...,13200,13300,13500,13700,14000,13800,13700,13700,13600,Algeria
3,1200,1200,1200,1200,1210,1210,1210,1210,1220,1220,...,41900,43700,44900,46600,48200,49800,51500,53200,55000,Andorra
4,618,620,623,626,628,631,634,637,640,642,...,6000,6190,6260,6230,6030,5940,5850,5760,5670,Angola
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,682,682,682,682,682,682,682,682,682,683,...,17700,17700,16700,15600,15000,14500,14200,14000,13600,Venezuela
189,861,861,861,861,861,861,861,861,862,862,...,4910,5120,5370,5670,5960,6250,6550,6870,7210,Vietnam
190,877,879,882,884,887,889,892,894,897,899,...,3790,3870,3770,2640,2330,2380,2430,2510,2560,Yemen
191,663,665,667,668,670,671,673,675,676,678,...,3510,3580,3630,3630,3640,3750,3870,3980,4110,Zambia


Βρίσκουμε το ελάχιστο και το μέγιστο κατακεφαλήν ΑΕΠ για όλες τις χώρες. Θα χρειαστούμε αυτές τις τιμές αργότερα για να ρυθμίσουμε τα όρια του διαγράμματός μας.

In [6]:
min_gdp = gdp_df.iloc[:,:-1].min().min() # minimum per rows, then minimum across rows
max_gdp = gdp_df.iloc[:,:-1].max().max() # maximum per rows, then maximum across rows
print(min_gdp, max_gdp)

247 178000


Θέλουμε να ομαδοποιήσουμε τις χώρες σύμφωνα με τις γεωγραφικές περιοχές τους. Οι γεωγραφικές περιοχές είναι από το <https://www.gapminder.org/fw/four-regions/> και συγκεκριμένα το <http://gapm.io/datageo>. Αφού διαβάσουμε τις περιοχές, τις ενώνουμε με τα στοιχεία του ΑΕΠ χρησιμοποιώντας τη στήλη `country`.

In [7]:
geographical_regions_df = pd.read_excel('Data Geographies - v2 - by Gapminder.xlsx',
                                        sheet_name='list-of-countries-etc')[['name', 'six_regions']]

all_df = pd.merge(gdp_df, geographical_regions_df, left_on='country', right_on='name')
all_df.sample(5, random_state=42)

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2014,2015,2016,2017,2018,2019,2020,country,name,six_regions
45,1920,1920,1920,1920,1920,1920,1920,1920,1920,1920,...,29100,30400,31100,31700,32300,33000,33700,Czech Republic,Czech Republic,europe_central_asia
137,1680,1680,1680,1680,1690,1690,1690,1690,1690,1690,...,26000,26500,27000,27500,27900,28300,28800,Portugal,Portugal,europe_central_asia
76,994,994,995,995,995,995,995,995,996,996,...,10000,10400,10800,11200,11700,12100,12600,Indonesia,Indonesia,east_asia_pacific
144,850,852,853,855,857,858,860,861,863,865,...,2890,2940,2990,3090,3190,3290,3390,Sao Tome and Principe,Sao Tome and Principe,sub_saharan_africa
113,1060,1060,1060,1060,1060,1060,1060,1060,1060,1060,...,14800,15300,15700,16200,16600,17000,17500,Montenegro,Montenegro,europe_central_asia


Στη συνέχεια παίρνουμε τα στοιχεία για το προσδόκιμο ζωής. Αυτός είναι ο δείκτης "Life expectancy (years)", διαθέσιμος στο <http://gapm.io/ilex>.

In [8]:
lex_df = pd.read_csv('lex.csv')

lex_df.sample(5, random_state=42)

,country,1800,1801,1802,1803,1804,1805,1806,1807,1808,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
139,Palau,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,79.2,79.4,79.6,79.7,79.9,80.1,80.2,80.4,80.6,80.7
113,Marshall Islands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,74.1,74.2,74.3,74.5,74.6,74.7,74.8,75.0,75.1,75.2
16,Bulgaria,35.8,35.8,35.9,35.9,36.0,36.0,36.1,36.1,36.1,...,83.7,83.8,83.9,84.0,84.2,84.3,84.4,84.5,84.6,84.7
75,Haiti,29.0,29.0,29.0,29.0,29.0,29.0,29.0,29.0,29.0,...,75.2,75.4,75.5,75.7,75.8,76.0,76.1,76.3,76.4,76.5
154,Solomon Islands,25.1,25.1,25.1,25.1,25.1,25.1,25.1,25.1,25.1,...,66.5,66.6,66.7,66.8,66.9,67.0,67.2,67.3,67.4,67.5


Βρίσκουμε και πάλι την ελάχιστη και τη μέγιστη τιμή, που θα τις χρειαστούμε για την προσαρμογή των ορίων του διαγράμματός μας.

In [9]:
min_lex = lex_df.iloc[:,1:].min().min()
max_lex = lex_df.iloc[:,1:].max().max()
print(min_lex, max_lex)

0.0 94.8


Ενώνουμε τα δεδομένα του προσδόκιμου ζωής με το `DataFrame` `all_df`, το οποίο σιγά-σιγά θα μεγαλώνει ώστε να περιέχει *όλα* μας τα δεδομένα. Επειδή υπάρχουν στήλες με τα ίδια ονόματα στα δύο `DataFrame` που ενώνουμε, θα δώσουμε εμείς τις δικές μας καταλήξεις για να έχουμε καλύτερα ονόματα από αυτά με τις καταλήξεις `_x` και `_y` που θα έβαζε το pandas μόνο του.

In [10]:
all_df = pd.merge(all_df, 
                  lex_df, 
                  on='country',
                  suffixes=("_gdp", "_lex"))

In [11]:
all_df.sample(5, random_state=42)

,1800_gdp,1801_gdp,1802_gdp,1803_gdp,1804_gdp,1805_gdp,1806_gdp,1807_gdp,1808_gdp,1809_gdp,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
183,682,682,682,682,682,682,682,682,682,683,...,87.3,87.4,87.5,87.7,87.8,87.9,88.1,88.2,88.3,88.4
162,518,518,518,518,519,519,519,519,519,519,...,79.4,79.5,79.7,79.8,80.0,80.1,80.3,80.4,80.6,80.7
18,597,597,597,597,597,597,597,597,597,597,...,75.1,75.3,75.4,75.5,75.6,75.8,75.9,76.0,76.1,76.2
15,608,608,608,609,609,609,610,610,610,611,...,85.2,85.3,85.5,85.6,85.7,85.8,85.9,86.0,86.1,86.3
67,857,857,857,857,858,858,858,858,858,858,...,83.6,83.7,83.8,83.9,84.1,84.2,84.3,84.4,84.5,84.6


Τέλος θα πρέπει να πάρουμε τα δεδομένα για τον πληθυσμό κάθε χώρας. Αυτός είναι ο δείκτης "Population, total" διαθέσιμος στο <http://gapm.io/dpop>. 

In [12]:
pop_df = pd.read_csv('pop.csv')
pop_df

,country,1800,1801,1802,1803,1804,1805,1806,1807,1808,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,Afghanistan,3.28M,3.28M,3.28M,3.28M,3.28M,3.28M,3.28M,3.28M,3.28M,...,124M,125M,126M,126M,127M,128M,128M,129M,130M,130M
1,Angola,1.57M,1.57M,1.57M,1.57M,1.57M,1.57M,1.57M,1.57M,1.57M,...,139M,140M,142M,143M,144M,145M,147M,148M,149M,150M
2,Albania,400k,402k,404k,405k,407k,409k,411k,413k,414k,...,1.34M,1.32M,1.3M,1.29M,1.27M,1.25M,1.23M,1.22M,1.2M,1.18M
3,Andorra,2650,2650,2650,2650,2650,2650,2650,2650,2650,...,52.8k,52.1k,51.5k,50.8k,50.2k,49.6k,49k,48.4k,47.8k,47.2k
4,UAE,40.2k,40.2k,40.2k,40.2k,40.2k,40.2k,40.2k,40.2k,40.2k,...,24.1M,24.3M,24.5M,24.7M,25M,25.2M,25.4M,25.7M,25.9M,26.1M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,Samoa,47.3k,47.3k,47.3k,47.3k,47.3k,47.3k,47.3k,47.2k,47.2k,...,370k,372k,374k,375k,377k,378k,380k,381k,382k,384k
193,Yemen,2.59M,2.59M,2.59M,2.59M,2.59M,2.59M,2.59M,2.59M,2.59M,...,107M,107M,107M,108M,108M,109M,109M,109M,110M,110M
194,South Africa,1.45M,1.45M,1.46M,1.46M,1.47M,1.47M,1.48M,1.49M,1.49M,...,92.4M,92.6M,92.9M,93.1M,93.3M,93.5M,93.7M,93.9M,94.1M,94.3M
195,Zambia,747k,758k,770k,782k,794k,806k,818k,831k,843k,...,61.1M,61.5M,61.9M,62.3M,62.7M,63.1M,63.4M,63.8M,64.1M,64.5M


Όπως και πριν, θα βγάλουμε τη στήλη `country`, θα κάνουμε τη μετατροπή των δεδομένων σε απλούς αριθμούς χωρίς καταλήξεις, και θα ξαναβάλουμε τη στήλη πίσω στο `DataFrame`.

In [13]:
pop_countries = pop_df.pop('country')
pop_df = pop_df.apply(suf2num, axis=0)
pop_df['country'] = pop_countries
pop_df

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2092,2093,2094,2095,2096,2097,2098,2099,2100,country
0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,...,125000000.0,126000000.0,126000000.0,127000000.0,128000000.0,128000000.0,129000000.0,130000000.0,130000000.0,Afghanistan
1,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,...,140000000.0,142000000.0,143000000.0,144000000.0,145000000.0,147000000.0,148000000.0,149000000.0,150000000.0,Angola
2,400000.0,402000.0,404000.0,405000.0,407000.0,409000.0,411000.0,413000.0,414000.0,416000.0,...,1320000.0,1300000.0,1290000.0,1270000.0,1250000.0,1230000.0,1220000.0,1200000.0,1180000.0,Albania
3,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,...,52100.0,51500.0,50800.0,50200.0,49600.0,49000.0,48400.0,47800.0,47200.0,Andorra
4,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,...,24300000.0,24500000.0,24700000.0,25000000.0,25200000.0,25400000.0,25700000.0,25900000.0,26100000.0,UAE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,47300.0,47300.0,47300.0,47300.0,47300.0,47300.0,47300.0,47200.0,47200.0,47200.0,...,372000.0,374000.0,375000.0,377000.0,378000.0,380000.0,381000.0,382000.0,384000.0,Samoa
193,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,...,107000000.0,107000000.0,108000000.0,108000000.0,109000000.0,109000000.0,109000000.0,110000000.0,110000000.0,Yemen
194,1450000.0,1450000.0,1460000.0,1460000.0,1470000.0,1470000.0,1480000.0,1490000.0,1490000.0,1500000.0,...,92600000.0,92900000.0,93100000.0,93300000.0,93500000.0,93700000.0,93900000.0,94100000.0,94300000.0,South Africa
195,747000.0,758000.0,770000.0,782000.0,794000.0,806000.0,818000.0,831000.0,843000.0,856000.0,...,61500000.0,61900000.0,62300000.0,62700000.0,63100000.0,63400000.0,63800000.0,64100000.0,64500000.0,Zambia


Θα αφαιρέσουμε τις προβολές.

In [14]:
pop_df = pop_df.drop([str(x) for x in range(2021, 2100+1)], axis=1)
pop_df

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,country
0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,3280000.0,...,30600000.0,31600000.0,32800000.0,33800000.0,34700000.0,35700000.0,36700000.0,37900000.0,39100000.0,Afghanistan
1,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,1570000.0,...,25200000.0,26200000.0,27200000.0,28200000.0,29200000.0,30200000.0,31300000.0,32400000.0,33500000.0,Angola
2,400000.0,402000.0,404000.0,405000.0,407000.0,409000.0,411000.0,413000.0,414000.0,416000.0,...,2910000.0,2910000.0,2900000.0,2900000.0,2900000.0,2900000.0,2890000.0,2890000.0,2870000.0,Albania
3,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,2650.0,...,76800.0,75200.0,73700.0,72200.0,72200.0,73800.0,75200.0,76500.0,77400.0,Andorra
4,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,40200.0,...,7500000.0,7830000.0,8240000.0,8670000.0,9030000.0,9230000.0,9350000.0,9380000.0,9450000.0,UAE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,47300.0,47300.0,47300.0,47300.0,47300.0,47300.0,47300.0,47200.0,47200.0,47200.0,...,196000.0,198000.0,200000.0,202000.0,203000.0,205000.0,208000.0,210000.0,212000.0,Samoa
193,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,2590000.0,...,28400000.0,29300000.0,30200000.0,31200000.0,32100000.0,33100000.0,34100000.0,35100000.0,36100000.0,Yemen
194,1450000.0,1450000.0,1460000.0,1460000.0,1470000.0,1470000.0,1480000.0,1490000.0,1490000.0,1500000.0,...,53800000.0,54700000.0,55600000.0,56700000.0,57300000.0,57600000.0,58600000.0,59600000.0,60600000.0,South Africa
195,747000.0,758000.0,770000.0,782000.0,794000.0,806000.0,818000.0,831000.0,843000.0,856000.0,...,14900000.0,15400000.0,15900000.0,16400000.0,16900000.0,17400000.0,18000000.0,18500000.0,19100000.0,Zambia


Τα ενώνουμε όλα μαζί.

In [15]:
all_df = pd.merge(all_df, 
                  pop_df, 
                  on='country')   
all_df.sample(10, random_state=42)

,1800_gdp,1801_gdp,1802_gdp,1803_gdp,1804_gdp,1805_gdp,1806_gdp,1807_gdp,1808_gdp,1809_gdp,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
183,682,682,682,682,682,682,682,682,682,683,...,29200000.0,29600000.0,29900000.0,30300000.0,30600000.0,30800000.0,30600000.0,29800000.0,28900000.0,28400000.0
162,518,518,518,518,519,519,519,519,519,519,...,36100000.0,36900000.0,37800000.0,38800000.0,40000000.0,41300000.0,42700000.0,44200000.0,45500000.0,46800000.0
18,597,597,597,597,597,597,597,597,597,597,...,10100000.0,10400000.0,10700000.0,11000000.0,11400000.0,11700000.0,12000000.0,12400000.0,12700000.0,13100000.0
15,608,608,608,609,609,609,610,610,610,611,...,9470000.0,9460000.0,9460000.0,9470000.0,9490000.0,9500000.0,9500000.0,9480000.0,9440000.0,9350000.0
67,857,857,857,857,858,858,858,858,858,858,...,14800000.0,15100000.0,15400000.0,15700000.0,16000000.0,16300000.0,16600000.0,16800000.0,17100000.0,17400000.0
108,518,518,518,519,519,519,520,520,520,521,...,108000.0,108000.0,108000.0,108000.0,109000.0,109000.0,109000.0,110000.0,110000.0,111000.0
45,1920,1920,1920,1920,1920,1920,1920,1920,1920,1920,...,10500000.0,10500000.0,10500000.0,10500000.0,10500000.0,10500000.0,10500000.0,10500000.0,10600000.0,10600000.0
76,994,994,995,995,995,995,995,995,996,996,...,249000000.0,253000000.0,256000000.0,259000000.0,262000000.0,265000000.0,267000000.0,270000000.0,272000000.0,275000000.0
16,2410,2410,2410,2410,2410,2410,2410,2410,2410,2410,...,11000000.0,11100000.0,11200000.0,11200000.0,11300000.0,11300000.0,11400000.0,11400000.0,11500000.0,11500000.0
146,1360,1360,1360,1360,1360,1360,1360,1360,1360,1360,...,7360000.0,7320000.0,7280000.0,7230000.0,7180000.0,7130000.0,7070000.0,7020000.0,6970000.0,6910000.0


Υπάρχει ένα λεπτό σημείο που πρέπει να φροντίσουμε. Το διάγραμμα να σχεδιαστεί μέσω JavaScript. Άρα τα ονόματα των στηλών του `DataFrame` θα πρέπει να είναι κατανοητά από την JavaScript. Τα ονόματα που στηλών που ξεκινούν με αριθμούς όμως δεν θα γίνουν αποδεκτά από την JavaScript, επομένως θα πρέπει να τα αναποδογυρίσουμε ώστε το `2020_gdp` ναι γίνει `gdp_2020`, το `2020_lex` να γίνει `lex_2020`, το `2020` να γίνει `pop_2020`, κ.λπ. για όλες τις χρονιές.

Ενόσω το κάνουμε αυτό, θα δημιουργήσουμε και μία ακόμα στήλη που θα χρησιμοποιήσουμε για το μέγεθος με το οποίο θα απεικονίσουμε κάθε χώρα. Το μέγεθος θα είναι ανάλογο με τον πληθυσμό, οπότε θα προσθέσουμε μια στήλη `size_x` (όπου `x` είναι η χρονιά). Επειδή το μέγεθος θα ορίζεται με βάση την ακτίνα του κύκλου για κάθε χώρα, θα πρέπει να πάρουμε την τετραγωνική ρίζα του πληθυσμού δια π.

Όλα αυτά θα γίνουν πιο γρήγορα αν βάλουμε τις νέες στήλες `size_x` σε ένα νεο `DataFrame` και καλέσουμε την `concat()`.

In [16]:
size_cols = {}
for column in all_df.columns:
    col = str(column) # needed for those columns whose name is a number (year)
    col_match = re.match(r'(\d+)(_(.+))?', col)
    if col_match:
        if col_match.lastindex > 1:
            new_name = col_match.group(3) + '_' + col_match.group(1)
            all_df.rename(columns={column: new_name},
                          inplace=True)
        else:
            new_name = 'pop_' + col_match.group(1)
            all_df.rename(columns={column: new_name},
                          inplace=True)
            sizes = 0.003 * np.sqrt(all_df[new_name] / np.pi)
            size_cols['size_' + col_match.group(1)] = sizes

all_df = pd.concat([all_df, pd.DataFrame(size_cols)], axis=1)
all_df

,gdp_1800,gdp_1801,gdp_1802,gdp_1803,gdp_1804,gdp_1805,gdp_1806,gdp_1807,gdp_1808,gdp_1809,...,size_2011,size_2012,size_2013,size_2014,size_2015,size_2016,size_2017,size_2018,size_2019,size_2020
0,603,603,603,603,603,603,603,603,603,603,...,9.161786,9.362828,9.514585,9.693559,9.840217,9.970365,10.113010,10.253670,10.419957,10.583631
1,667,667,667,667,667,668,668,668,668,668,...,2.887306,2.887306,2.887306,2.882341,2.882341,2.882341,2.882341,2.877367,2.877367,2.867393
2,715,716,717,718,719,720,721,722,723,724,...,10.281572,10.378635,10.488465,10.597157,10.704745,10.824503,10.929854,11.034198,11.137565,11.227231
3,1200,1200,1200,1200,1210,1210,1210,1210,1220,1220,...,0.472102,0.469058,0.464147,0.459494,0.454794,0.454794,0.459806,0.464147,0.468141,0.470887
4,618,620,623,626,628,631,634,637,640,642,...,8.326337,8.496628,8.663572,8.827359,8.988162,9.146138,9.301431,9.469313,9.634270,9.796450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,682,682,682,682,682,682,682,682,682,683,...,9.146138,9.208570,9.255117,9.316818,9.362828,9.393375,9.362828,9.239627,9.099033,9.019978
184,861,861,861,861,861,861,861,861,862,862,...,15.922746,16.012452,16.110552,16.208058,16.304981,16.410063,16.514476,16.600985,16.687046,16.764122
185,877,879,882,884,887,889,892,894,897,899,...,8.892029,9.019978,9.161786,9.301431,9.454175,9.589563,9.737788,9.883790,10.027666,10.169507
186,663,665,667,668,670,671,673,675,676,678,...,6.422847,6.533403,6.642119,6.749085,6.854381,6.958084,7.060264,7.180961,7.280013,7.397126


Κάθε χώρα θα εμφανιστεί με χρώμα που αντιστοιχεί στη γεωγραφική της περιοχή. Για να γίνει αυτό, θα πρέπει να πάρουμε έναν αριθμητικό κωδικό για κάθε χώρα.

In [17]:
all_df['Region Cat'] = all_df['six_regions'].astype('category')

Το διάγραμμα θα δείχνει το προσδόκιμο ζωής, κατακεφαλήν ΑΕΠ, και πληθυσμό για κάθε χρόνο, τον οποίο θα επιλέγουμε αλληλεπιδραστικά. Θα προσθέσουμε στο `DataFrame` μας:

* μια στήλη `x` με το κατακεφαλήν ΑΕΠ για την επιλεγμένη χρονιά

* μια στήλη `y` με το προσδόκιμο ζωής για την επιλεγμένη χρονιά

* μια στήλη `pop` με τον πληθυσμό για την επιλεγμένη χρονιά

* μια στήλη `size` με το μέγεθος της χώρας για την επιλεγμένη χρονιά

Επίσης θα συγκεντρώσουμε τα χρώματα της κάθε χώρας. Αυτά θα προκύψουν από τη χρωματική παλέτα (`Spectral6` αφού έχουμε έξι περιοχές) και τον κωδικό γεωγραφικής περιοχής κάθε χώρας.

In [18]:
from bokeh.palettes import Spectral6

all_df['x'] = all_df['gdp_2020']
all_df['y'] = all_df['lex_2020']
all_df['pop'] = all_df['pop_2020']

all_df['size'] = all_df['size_2020']
all_df['colors'] = all_df['Region Cat'].cat.codes.map(lambda x: Spectral6[x])

Τα υπόλοιπα είναι θέμα Bokeh και JavaScript.

In [19]:
import bokeh.plotting as bk

from bokeh.models import (ColumnDataSource,
                          HoverTool, 
                          BoxZoomTool,
                          ResetTool,
                          PanTool,
                          WheelZoomTool,
                          CustomJS,
                          NumeralTickFormatter)

from bokeh.models.widgets import Slider

from bokeh.layouts import column

dim = 1000 # width and length

source = ColumnDataSource(all_df)

hover = HoverTool(tooltips=[
        ("Country Name", "@country"),
        ("Population", "@pop")
        ]
    )

min_x = 100 * (min_gdp // 100)
max_x = 1000 * (max_gdp // 1000) + 1000

min_y = 10 * (min_lex // 10)
max_y = 100 * (max_lex // 100) + 100

tools = [
    hover,
    WheelZoomTool(),
    PanTool(),
    BoxZoomTool(),
    ResetTool()
]

p = bk.figure(tools=tools,
              x_axis_type="log",
              x_axis_label="Income per person "
              "(GDP/capita, PPP$ inflation-adjusted)",
              y_axis_label="Life expectancy (years)",
              x_range=(min_x, max_x), 
              y_range=(min_y, max_y),
              width=dim,
              height=dim)

p.xaxis[0].formatter = NumeralTickFormatter(format='0a')

text_x = 15000
text_y = 20

regions = all_df['Region Cat'].cat.categories
for i, region in enumerate(regions):
    # Add text labels using the text() method
    p.text(x=[text_x],
           y=[text_y],
           text=[region],
           text_font_size='10pt',
           text_color='#666666')
    
    # Add circle markers using the scatter() method
    p.scatter(x=[text_x - 2000],
             y=[text_y + 1],
             fill_color=Spectral6[i % len(Spectral6)],
             size=10,
             line_color=None, 
             fill_alpha=0.8,
             marker='circle')
    
    text_y = text_y - 3

callback = CustomJS(args=dict(source=source), code="""
        var data = source.data;
        var v = cb_obj.value;
        var x = data['gdp_' + v];
        var y = data['lex_' + v];
        data['x'] = x;
        data['y'] = y;
        // As we do not have data for size and population
        // for all year, go to the nearest year for which
        // we do have data.
        if (!data['size_' + v]) {
          v = Math.ceil(v / 10) * 10;
        }
        data['size'] = data['size_' + v];
        data['pop'] = data['pop_' + v];
        source.change.emit();
        // source.trigger('change');
    """)

p.scatter(x='x', 
          y='y',
          size='size',
          fill_color='colors',
          fill_alpha=0.8,
          source=source)

slider = Slider(start=1800, end=2020, value=2020, step=1, title="Year", width=dim)
slider.js_on_change('value', callback)


bk.output_notebook()
bk.output_file('gdp_life_expectancy_gapminder.html')
#pg = gridplot([p, slider], ncols=1, sizing_mode='stretch_both')
#bk.show(pg)
bk.show(column(p, slider))

Loading BokehJS ...